# EXP-02 live eval — ingest `kb_ddxplus.json`, retrieve, compare to experiment

End-to-end check that the **production RAG stack** (OpenSearch + BGE + hybrid RRF) matches
[EXP-02](../experiments/exp02/README.md) offline benchmarks.

## Pipeline
1. Load `data/kb/kb_ddxplus.json` → `DiseaseDocument`
2. Ingest via `RAGService` (embed + bulk upsert)
3. Build patient queries exactly like `experiments/exp02/eval_retrieval.py`
4. Retrieve with `Retriever` (BM25 + dense kNN + hybrid RRF)
5. Report Hit@1 / Hit@3 / Hit@5 / MRR vs committed baselines

## Prerequisites
- `.env` with `OPENSEARCH_*` credentials
- Index migrated: `uv run python -m src.migrations.migrate_ddxplus_index upgrade`
- DDXPlus validate patients (not in repo — licensed data):
  - Download `release_validate_patients` from DDXPlus Figshare
  - Place at `data/eval/release_validate_patients` (CSV)
  - Place `release_evidences.json` at `data/eval/release_evidences.json`
- Optional: `export EXP02_DATA=/path/to/eval/inputs`

## Performance / OpenSearch client

The sync client uses **urllib3** (`Urllib3HttpConnection`). For parallel eval:

| Setting | Default | Notes |
|---------|---------|-------|
| `EXP02_WORKERS` | `8` | Concurrent searches via `asyncio.to_thread` |
| `OPENSEARCH_POOL_MAXSIZE` | `16` in `.env` | Must be ≥ `EXP02_WORKERS` or urllib3 logs `Connection pool is full` |
| `OPENSEARCH_TIMEOUT` | `60` | Raise for slow remote bulk/search |
| `EXP02_SAMPLE` | `5000` | `132448` = full validate set |

Restart the kernel after changing `OPENSEARCH_*` pool/timeout settings (client singleton).

Query construction uses `normalize_symptom_phrase` from `src.services.rag.preprocess` — same
normalizer as `build_kb.py`. Preprocessing is **disabled** on the retriever (`Exp02Preprocess`)
so queries match EXP-02 BM25/hybrid baselines.

## Baseline parity (what matches offline EXP-02)

| Mode | Fair comparison? | Why |
|------|------------------|-----|
| **BM25** | Yes | Same space-joined phrase text; live uses OpenSearch `match` on `keyword_text` |
| **Hybrid RRF** | Yes | Same query text; live uses OpenSearch RRF pipeline vs offline local fusion |
| **Dense kNN** | **Approximate only** | Offline `eval_dense_hybrid.py` uses legacy `embed_text_query()` — see Notes |

**Doc embeddings** (ingest) match offline no-desc: `build_embed_text` via `DiseaseDocument`, no description in the vector.

**Query embeddings** for vector/hybrid use production `TextEmbeddingService.embed_query()` / `embed_queries()` (space-joined phrases + BGE query prefix). Offline dense uses a different template — do not expect Hit@1 to match the 83.11% dense baseline closely.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "services" / "rag").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists(), "Run from repo root or notebooks/"
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: /home/mitu/projects/disease-diagnosis-rag-system


In [2]:
import ast
import csv
import json
import random
import zipfile
from collections import defaultdict
from dataclasses import dataclass

from src.db.vector_db.opensearch import get_opensearch_client
from src.services.inference.embeddings.service import TextEmbeddingService
from src.services.rag.pipeline import RAGService
from src.services.rag.preprocess import PreprocessPipeline, normalize_symptom_phrase
from src.services.rag.retrieve import Retriever
from src.services.rag.schemas import (
    Bm25RetrieveRequest,
    DiseaseDocument,
    HybridRetrieveRequest,
    VectorRetrieveRequest,
)
from src.settings import settings

# --- eval config (match experiments/exp02) ---
SAMPLE = int(os.environ.get("EXP02_SAMPLE", "5000"))  # 132448 = full validate set
SEED = 13
EVAL_TOP_K = 49  # full corpus — needed for correct Hit@5 / MRR
EVAL_WORKERS = int(os.environ.get("EXP02_WORKERS", "8"))  # keep <= OPENSEARCH_POOL_MAXSIZE
REINGEST = False  # set False to skip ingest if index already loaded

KB_PATH = PROJECT_ROOT / "data/kb/kb_ddxplus.json"
BASELINE_BM25 = PROJECT_ROOT / "experiments/exp02/results/EXP02_eval_report.md"
BASELINE_HYBRID = PROJECT_ROOT / "experiments/exp02/results/dense_hybrid_summary.md"

EXP02_DATA = Path(os.environ.get("EXP02_DATA", PROJECT_ROOT / "data/eval"))
EVID_PATH = EXP02_DATA / "release_evidences.json"
PATIENTS_PATH = EXP02_DATA / "release_validate_patients"

for zip_path in (
    EXP02_DATA / "release_validate_patients.zip",
    EXP02_DATA / "eval" / "release_validate_patients.zip",
):
    if not PATIENTS_PATH.exists() and zip_path.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(EXP02_DATA)
        break

print("Index alias:", settings.RETRIEVE_INDEX_ALIAS)
print("Search pipeline:", settings.CURRENT_SEARCH_PIPELINE)
print("KB:", KB_PATH)
print("Patients:", PATIENTS_PATH, "exists:", PATIENTS_PATH.exists())
print("Evidences:", EVID_PATH, "exists:", EVID_PATH.exists())
print(f"Sample size: {SAMPLE} (SEED={SEED})")

/home/mitu/projects/disease-diagnosis-rag-system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index alias: diseases
Search pipeline: hybrid-rrf
KB: /home/mitu/projects/disease-diagnosis-rag-system/data/kb/kb_ddxplus.json
Patients: /home/mitu/projects/disease-diagnosis-rag-system/data/eval/release_validate_patients exists: True
Evidences: /home/mitu/projects/disease-diagnosis-rag-system/data/eval/release_evidences.json exists: True
Sample size: 5000 (SEED=13)


## Step 1 — Load KB and ingest

Loads all 49 documents from `kb_ddxplus.json` and upserts through `RAGService.ingest()`.

In [3]:
def kb_row_to_document(row: dict) -> DiseaseDocument:
    return DiseaseDocument(
        doc_id=row["doc_id"],
        disease=row["disease"],
        symptoms=row["symptoms"],
        antecedents=row["antecedents"],
        severity=row["severity"],
        description=row["description"],
        source=row["source"],
    )


kb_rows = json.loads(KB_PATH.read_text(encoding="utf-8"))
records = [kb_row_to_document(row) for row in kb_rows]
disease_to_doc_id = {row["disease"]: row["doc_id"] for row in kb_rows}

print(f"Loaded {len(records)} KB documents")
print(f"Sample doc_id: {records[0].doc_id} — embed_text[:80]: {records[0].embed_text[:80]}...")

embed_service = TextEmbeddingService()
rag = RAGService(embed_service=embed_service)
if REINGEST:
    count = rag.ingest(records)
    print(f"Ingested {count} docs into alias '{settings.RETRIEVE_INDEX_ALIAS}'")
else:
    print("Skipping ingest (REINGEST=False)")

Loaded 49 KB documents
Sample doc_id: J93 — embed_text[:80]: Disease: Spontaneous pneumothorax. Symptoms: pain present, pain related to consu...
Skipping ingest (REINGEST=False)


In [4]:
client = get_opensearch_client()
body = {"size": 0, "track_total_hits": True, "query": {"match_all": {}}}
resp = client.query(settings.RETRIEVE_INDEX_ALIAS, body)
total = resp.hits.total.value if hasattr(resp.hits.total, "value") else int(resp.hits.total)
print(f"Documents in '{settings.RETRIEVE_INDEX_ALIAS}': {total} (expected 49)")

Documents in 'diseases': 50 (expected 49)


## Step 2 — Build EXP-02 patient queries

Same query construction as `experiments/exp02/eval_retrieval.py`:
evidence codes → `normalize_symptom_phrase(question_en)` → space-joined tokens.

In [5]:
if not PATIENTS_PATH.exists() or not EVID_PATH.exists():
    raise FileNotFoundError(
        "DDXPlus eval data missing. See notebook intro — need release_validate_patients "
        f"and release_evidences.json under {EXP02_DATA}"
    )

evidences = json.loads(EVID_PATH.read_text(encoding="utf-8"))
code_phrase = {
    code: normalize_symptom_phrase(meta.get("question_en", code))
    for code, meta in evidences.items()
}


def base_code(evidence: str) -> str:
    return evidence.split("_@_")[0]


def patient_query_text(evidences_field: str) -> str:
    try:
        items = ast.literal_eval(evidences_field)
    except (SyntaxError, ValueError):
        items = []
    phrases: list[str] = []
    seen: set[str] = set()
    for ev in items:
        phrase = code_phrase.get(base_code(str(ev)))
        if phrase and phrase not in seen:
            seen.add(phrase)
            phrases.append(phrase)
    return " ".join(phrases)


with PATIENTS_PATH.open(encoding="utf-8") as handle:
    patient_rows = list(csv.DictReader(handle))

if SAMPLE and SAMPLE < len(patient_rows):
    random.seed(SEED)
    patient_rows = random.sample(patient_rows, SAMPLE)

eval_cases = []
skipped = 0
for row in patient_rows:
    gold = row["PATHOLOGY"]
    if gold not in disease_to_doc_id:
        continue
    query = patient_query_text(row["EVIDENCES"])
    if not query.strip():
        skipped += 1
        continue
    eval_cases.append({"gold_disease": gold, "query": query})

print(f"Eval cases: {len(eval_cases)} (skipped empty queries: {skipped})")
print(f"Example query: {eval_cases[0]['query'][:120]}...")

Eval cases: 5000 (skipped empty queries: 0)
Example query: significantly increased sweating pain related to consultation pain character pain present pain intensity pain radiation ...


## Step 3 — Retrieve and score

Uses `Retriever` with identity preprocessing (EXP-02 queries are already normalized).

Baselines (full validate, n=132448) from `experiments/exp02/results/`:

| Mode | Hit@1 | MRR | Parity with live |
|------|-------|-----|------------------|
| BM25 (`keyword_text`) | 98.47% | 0.9921 | Direct |
| Dense kNN (no-desc embed) | 83.11% | 0.8911 | **Legacy query embed** — see below |
| Hybrid RRF (no-desc embed) | 91.17% | 0.9511 | Direct (BM25-heavy on this eval) |

**Dense baseline caveat:** offline `eval_dense_hybrid.py` embeds queries with `embed_text_query()`:

```text
Symptoms and antecedents: fever, cough, fatigue.
```

(no BGE query prefix). Live vector/hybrid uses production `embed_query()` on space-joined phrases **with** the BGE prefix (`Represent this sentence for searching relevant passages: ...`). Doc side matches; query side does not — treat dense offline numbers as directional only.

In [6]:
import asyncio


class Exp02Preprocess(PreprocessPipeline):
    """EXP-02 queries are pre-normalized; skip preprocess_query."""

    def preprocess_query(self, query: str) -> str:
        return query


@dataclass
class EvalMetrics:
    hit1: int = 0
    hit3: int = 0
    hit5: int = 0
    rr_sum: float = 0.0
    n: int = 0

    def add_rank(self, rank: int | None) -> None:
        if rank is None:
            return
        self.n += 1
        if rank == 1:
            self.hit1 += 1
        if rank <= 3:
            self.hit3 += 1
        if rank <= 5:
            self.hit5 += 1
        self.rr_sum += 1.0 / rank

    def as_dict(self) -> dict[str, float | int]:
        if self.n == 0:
            return {"n": 0, "hit@1": 0.0, "hit@3": 0.0, "hit@5": 0.0, "mrr": 0.0}
        return {
            "n": self.n,
            "hit@1": self.hit1 / self.n,
            "hit@3": self.hit3 / self.n,
            "hit@5": self.hit5 / self.n,
            "mrr": self.rr_sum / self.n,
        }


def rank_of_gold(hits, gold_disease: str) -> int | None:
    for hit in hits:
        if hit.disease == gold_disease:
            return hit.rank
    return None


def _eval_one(retriever: Retriever, mode: str, case: dict, embedding: list[float] | None) -> tuple[str, int | None]:
    query = case["query"]
    gold = case["gold_disease"]
    if mode == "bm25":
        result = retriever.search_bm25(Bm25RetrieveRequest(query=query, top_k=EVAL_TOP_K))
    elif mode == "vector":
        result = retriever.search_vector(
            VectorRetrieveRequest(query=query, top_k=EVAL_TOP_K, embedding=embedding)
        )
    elif mode == "hybrid":
        result = retriever.search_hybrid(
            HybridRetrieveRequest(query=query, top_k=EVAL_TOP_K, embedding=embedding)
        )
    else:
        raise ValueError(mode)
    return gold, rank_of_gold(result.hits, gold)


async def run_eval(retriever: Retriever, mode: str, cases: list[dict]) -> tuple[EvalMetrics, dict[str, EvalMetrics]]:
    """Run eval with asyncio.to_thread (same pattern as future FastAPI routes)."""
    metrics = EvalMetrics()
    per_disease: dict[str, EvalMetrics] = defaultdict(EvalMetrics)
    semaphore = asyncio.Semaphore(EVAL_WORKERS)

    embeddings: list[list[float] | None]
    if mode in ("vector", "hybrid"):
        print(f"  {mode}: embedding {len(cases)} queries (batched) ...")
        embeddings = await asyncio.to_thread(
            embed_service.embed_queries, [case["query"] for case in cases]
        )
    else:
        embeddings = [None] * len(cases)

    async def eval_one(case: dict, embedding: list[float] | None) -> tuple[str, int | None]:
        async with semaphore:
            return await asyncio.to_thread(_eval_one, retriever, mode, case, embedding)

    tasks = [
        asyncio.create_task(eval_one(case, embedding))
        for case, embedding in zip(cases, embeddings, strict=True)
    ]

    completed = 0
    for task in asyncio.as_completed(tasks):
        gold, rank = await task
        metrics.add_rank(rank)
        per_disease[gold].add_rank(rank)
        completed += 1
        if completed % 500 == 0:
            print(f"  {mode}: {completed}/{len(cases)} ...")

    return metrics, per_disease


def pct(value: float) -> str:
    return f"{100.0 * value:.2f}%"


retriever = Retriever(
    client=client,
    embed_service=embed_service,
    preprocess=Exp02Preprocess(),
)
print(f"Retriever ready (EXP-02 query mode, workers={EVAL_WORKERS})")

Retriever ready (EXP-02 query mode, workers=8)


In [7]:
print("Running BM25 eval ...")
bm25_metrics, bm25_per_disease = await run_eval(retriever, "bm25", eval_cases)

print("Running dense (kNN) eval ...")
vector_metrics, vector_per_disease = await run_eval(retriever, "vector", eval_cases)

print("Running hybrid eval ...")
hybrid_metrics, hybrid_per_disease = await run_eval(retriever, "hybrid", eval_cases)

results = {
    "BM25 (live OpenSearch)": bm25_metrics.as_dict(),
    "Dense kNN (live OpenSearch)": vector_metrics.as_dict(),
    "Hybrid RRF (live OpenSearch)": hybrid_metrics.as_dict(),
}

baselines = {
    "BM25 (EXP-02 offline)": {"n": 132448, "hit@1": 0.9847, "hit@3": 0.9995, "hit@5": 0.9999, "mrr": 0.9921},
    "Dense kNN (EXP-02 offline, legacy query embed)": {
        "n": 132448,
        "hit@1": 0.8311,
        "hit@3": 0.9441,
        "hit@5": 0.9682,
        "mrr": 0.8911,
    },
    "Hybrid RRF (EXP-02 offline)": {"n": 132448, "hit@1": 0.9117, "hit@3": 0.9889, "hit@5": 0.9961, "mrr": 0.9511},
}

print("\n=== Live vs EXP-02 baseline ===")
print(f"{'config':<32} {'Hit@1':>8} {'Hit@3':>8} {'Hit@5':>8} {'MRR':>8} {'n':>8}")
for label, row in {**results, **baselines}.items():
    print(
        f"{label:<32} {pct(row['hit@1']):>8} {pct(row['hit@3']):>8} "
        f"{pct(row['hit@5']):>8} {row['mrr']:>8.4f} {row['n']:>8}"
    )

Running BM25 eval ...
  bm25: 500/5000 ...
  bm25: 1000/5000 ...
  bm25: 1500/5000 ...
  bm25: 2000/5000 ...
  bm25: 2500/5000 ...
  bm25: 3000/5000 ...
  bm25: 3500/5000 ...
  bm25: 4000/5000 ...
  bm25: 4500/5000 ...
  bm25: 5000/5000 ...
Running dense (kNN) eval ...
  vector: embedding 5000 queries (batched) ...
2026-06-25T17:18:10.682290Z [info     ] No device provided, using cuda:0 [sentence_transformers.base.model]
2026-06-25T17:18:10.691815Z [info     ] Loading SentenceTransformer model from /home/mitu/projects/disease-diagnosis-rag-system/models/bge-small-en-v1.5. [sentence_transformers.base.model]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2050.23it/s]


  vector: 500/5000 ...
  vector: 1000/5000 ...
  vector: 1500/5000 ...
  vector: 2000/5000 ...
  vector: 2500/5000 ...
  vector: 3000/5000 ...
  vector: 3500/5000 ...
  vector: 4000/5000 ...
  vector: 4500/5000 ...
  vector: 5000/5000 ...
Running hybrid eval ...
  hybrid: embedding 5000 queries (batched) ...
  hybrid: 500/5000 ...
  hybrid: 1000/5000 ...
  hybrid: 1500/5000 ...
  hybrid: 2000/5000 ...
  hybrid: 2500/5000 ...
  hybrid: 3000/5000 ...
  hybrid: 3500/5000 ...
  hybrid: 4000/5000 ...
  hybrid: 4500/5000 ...
  hybrid: 5000/5000 ...

=== Live vs EXP-02 baseline ===
config                              Hit@1    Hit@3    Hit@5      MRR        n
BM25 (live OpenSearch)             98.86%   99.96%   99.98%   0.9941     5000
Dense kNN (live OpenSearch)        79.38%   93.66%   96.92%   0.8701     5000
Hybrid RRF (live OpenSearch)       92.50%   98.86%   99.52%   0.9575     5000
BM25 (EXP-02 offline)              98.47%   99.95%   99.99%   0.9921   132448
Dense kNN (EXP-02 offline, n

In [8]:
AFFECTED = {
    "Anemia", "Bronchiectasis", "Cluster headache", "GERD",
    "Guillain-Barré syndrome", "Localized edema", "Panic attack",
    "Pulmonary embolism", "Tuberculosis",
}

print("=== Lowest BM25 Hit@1 diseases (live) ===")
rows = []
for disease, metrics in bm25_per_disease.items():
    d = metrics.as_dict()
    if d["n"]:
        rows.append((d["hit@1"], disease, d["n"], d["mrr"]))
rows.sort()
for hit1, disease, n, mrr in rows[:10]:
    flag = "⚠️" if disease in AFFECTED else ""
    print(f"{pct(hit1):>7}  MRR={mrr:.4f}  n={n:>5}  {disease} {flag}")

=== Lowest BM25 Hit@1 diseases (live) ===
 64.29%  MRR=0.8214  n=   84  Acute rhinosinusitis 
 82.50%  MRR=0.9083  n=   80  Unstable angina 
 97.79%  MRR=0.9880  n=  362  URTI 
 98.08%  MRR=0.9904  n=   52  Myocarditis 
 98.35%  MRR=0.9917  n=  121  Acute laryngitis 
 99.28%  MRR=0.9942  n=  139  Bronchitis 
 99.70%  MRR=0.9985  n=  336  Viral pharyngitis 
100.00%  MRR=1.0000  n=   83  Acute COPD exacerbation / infection 
100.00%  MRR=1.0000  n=  103  Acute dystonic reactions 
100.00%  MRR=1.0000  n=  134  Acute otitis media 


## Notes

- **Sample vs full**: default `SAMPLE=5000` with `SEED=13` matches EXP-02 sampling. For full
  validate (132448 patients) set `EXP02_SAMPLE=132448`.
- **Speed**: vector/hybrid pre-embed all queries in batches (`embed_queries`), then run OpenSearch
  searches concurrently via `asyncio.to_thread` + semaphore (`EXP02_WORKERS=8`).
- **BM25 gap**: live OpenSearch BM25 should be close to offline EXP-02 (~98.5% Hit@1 on full set).
  Small gaps can come from analyzer differences vs the offline Okapi implementation.
- **Hybrid gap**: live hybrid uses OpenSearch RRF pipeline; offline EXP-02 fuses ranks locally.
  Both use the same BM25 query text and no-desc doc embeddings — close match is expected.
- **Dense kNN gap (expected)**: offline `eval_dense_hybrid.py` does **not** use production embedding:

  | | Offline `embed_text_query()` | Live `embed_query()` / `embed_queries()` |
  |--|------------------------------|------------------------------------------|
  | Query string | `Symptoms and antecedents: {phrases}.` | `{phrase1} {phrase2} ...` (space-joined) |
  | BGE query prefix | No | Yes (`BGE_QUERY_PREFIX`) |
  | Doc embed | no-desc `build_embed_text` (local numpy) | no-desc `build_embed_text` at ingest (OpenSearch kNN) |

  A live dense Hit@1 ~4 pp below the offline 83.11% baseline is **not** a production bug by itself.
  Use BM25 + hybrid to validate the stack; re-baseline dense with `embed_query()` offline if you need strict parity.
- **Re-ingest**: set `REINGEST=False` if the index already contains the 49 KB docs.